В качестве исходного кода для ускорения возьмём задачу 7.2 (об уравнении Бюргерса).

In [33]:
import numpy as np
import time
from numba import njit, prange

In [34]:
def flux(u):
    return 0.5 * u ** 2

def upwind_sim(u, dt, dx, time_steps):
    for i in range(time_steps):
        f = flux(u[i, :])
        u[i+1, :] = u[i, :] - (dt / dx) * np.where(u[i, :] > 0, f - np.roll(f, 1), np.roll(f, -1) - f)
    return u

def lax_wendroff_sim(u, dt, dx, time_steps):
    for i in range(time_steps):
        f = flux(u[i, :])
        f_plus  = np.roll(f, -1)
        f_minus = np.roll(f, 1)
        a_plus  = 0.5 * (u[i, :] + np.roll(u[i, :], -1))
        a_minus = 0.5 * (u[i, :] + np.roll(u[i, :], 1))
        u[i+1, :] = u[i, :] - (dt / (2.0 * dx)) * (f_plus - f_minus) + (dt ** 2) / (2.0 * dx ** 2) * (a_plus * (f_plus - f) - a_minus * (f - f_minus))
    return u

In [35]:
def initial_condition(x):
    return np.sin(2*np.pi*x)

In [36]:
n = 1000
x = np.linspace(0.0, 1.0, n)
dx = x[1] - x[0]
dt = 0.001
time_steps = 2500

u_uw = np.zeros((time_steps + 1, n))
u_uw[0, :] = initial_condition(x)
u_lw = np.zeros((time_steps + 1, n))
u_lw[0, :] = initial_condition(x)

In [43]:
elapsed_times = np.zeros(10)
for i in range(10):
    elapsed_times[i] = time.time()
    u_uw = upwind_sim(u_uw, dt, dx, time_steps)
    u_lw = lax_wendroff_sim(u_lw, dt, dx, time_steps)
    elapsed_times[i] = time.time() - elapsed_times[i]
elapsed_time1 = np.average(elapsed_times)
print('Среднее время работы без параллелизации =', elapsed_time1, 'сек')

Среднее время работы без параллелизации = 0.14994735717773439 сек


In [47]:
@njit
def upwind_sim_parallel(u, dt, dx, time_steps):
    nx = u.shape[1]
    for t in prange(time_steps):
        for j in range(nx):
            uj = u[t, j]
            fj = 0.5 * uj * uj
            j_left  = j - 1 if j > 0 else nx - 1
            j_right = j + 1 if j < nx - 1 else 0
            f_left  = 0.5 * u[t, j_left] * u[t, j_left]
            f_right = 0.5 * u[t, j_right] * u[t, j_right]
            if uj > 0:
                df = fj - f_left
            else:
                df = f_right - fj
            u[t+1, j] = uj - (dt / dx) * df
    return u

@njit
def lax_wendroff_sim_parallel(u, dt, dx, time_steps):
    nx = u.shape[1]
    for t in prange(time_steps):
        for j in range(nx):
            j_left  = j - 1 if j > 0 else nx - 1
            j_right = j + 1 if j < nx - 1 else 0
            u0  = u[t, j]
            ul  = u[t, j_left]
            ur  = u[t, j_right]
            f0  = 0.5 * u0 * u0
            fl  = 0.5 * ul * ul
            fr  = 0.5 * ur * ur
            a_plus  = 0.5 * (u0 + ur)
            a_minus = 0.5 * (u0 + ul)
            u[t+1, j] = u0 - (fr - fl) * (dt / (2 * dx)) + (dt*dt) / (2 * dx * dx) * (a_plus*(fr - f0) - a_minus*(f0 - fl))

    return u

В исходном варианте вычисления сильно тормозились из-за использования np.roll и np.where, которые заставляли создавать скрытые временные массивы и мешали нормальному распараллеливанию циклов. Код был переписан так, чтобы не полагаться на эти операции. Сдвиги были заменены ручной индексацией соседних ячеек, а выбор направления потока -- простым условием if-else.


In [48]:

elapsed_times = np.zeros(10)
for i in range(10):
    elapsed_times[i] = time.time()
    u_uw = upwind_sim_parallel(u_uw, dt, dx, time_steps)
    u_lw = lax_wendroff_sim_parallel(u_lw, dt, dx, time_steps)
    elapsed_times[i] = time.time() - elapsed_times[i]
elapsed_time2 = np.average(elapsed_times)
print('Среднее время работы с параллелизацией =', elapsed_time2, 'сек')

Среднее время работы с параллелизацией = 0.028333067893981934 сек


In [49]:
print('Получили ускорение в', elapsed_time1/elapsed_time2, 'раз')

Получили ускорение в 5.2923092458188705 раз
